In [1]:
from __future__ import annotations

import re
import time
from urllib.parse import urlparse
from datetime import datetime
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import pandas as pd
from tqdm.auto import tqdm

import re
from rapidfuzz import process, fuzz

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# -----------------------------
# Helpers
# -----------------------------
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def norm_name(name: str) -> str:
    s = re.sub(r"[^\w\s]", " ", str(name).lower())
    parts = [p for p in s.split() if p not in SUFFIXES]
    return " ".join(parts)

def classify_fbs(team_text: Optional[str]) -> Optional[str]:
    # If you have your own P4/G5 map, plug it here.
    # Otherwise leave blank and/or fill later.
    if not team_text:
        return None
    return None

def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

@dataclass
class Player247:
    name: str
    id_247: Optional[int] = None

    position: Optional[str] = None

    hs_name: Optional[str] = None
    hs_city: Optional[str] = None
    hs_state: Optional[str] = None
    hs_class: Optional[str] = None  # "Class 2021"
    hs_exp: Optional[str] = None    # "Exp 2021 - 2024" (college page)

    hs_stars: Optional[int] = None
    hs_rating_247: Optional[float] = None
    hs_natl_rank: Optional[int] = None
    hs_pos_rank: Optional[int] = None

    transfer_year: Optional[int] = None
    transfer_origin: Optional[str] = None
    transfer_destination: Optional[str] = None
    transfer_stars: Optional[int] = None
    transfer_rating: Optional[float] = None   # could be 0.9200 or 92 depending on page block
    transfer_ovr_rank: Optional[int] = None
    transfer_pos_rank: Optional[int] = None

    source_player_url: Optional[str] = None
    source_hs_url: Optional[str] = None


In [2]:
# -----------------------------
# Selenium setup
# -----------------------------
def make_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    # helps reduce some bot friction
    opts.add_argument("--lang=en-US")
    driver = webdriver.Chrome(options=opts)  # Selenium Manager will fetch driver if needed
    driver.set_page_load_timeout(45)
    return driver


In [3]:
driver = make_driver(headless=True)

In [5]:
# -----------------------------
# 247 scraping primitives
# -----------------------------
def click_load_more_until_done(
    driver: webdriver.Chrome,
    timeout: int = 12,
    max_clicks: int = 200
) -> None:
    for _ in tqdm(range(max_clicks), desc="Clicking 'Load More'", unit="click"):
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[contains(., 'Load More Players') or contains(., 'Load More')]"
                ))
            )
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)
            btn.click()
            time.sleep(0.8)  # let items render
        except Exception:
            break


def scrape_portal_player_links(
    driver: webdriver.Chrome,
    portal_url: str
) -> List[str]:
    driver.get(portal_url)
    time.sleep(1.5)

    click_load_more_until_done(driver)

    anchors = driver.find_elements(
        By.XPATH,
        "//a[contains(@href, '/player/') and not(contains(@href, '#'))]"
    )

    urls = []
    for a in tqdm(anchors, desc="Collecting player links", unit="link"):
        href = a.get_attribute("href")
        if href and "/player/" in href:
            urls.append(href.split("?")[0].rstrip("/"))

    # dedupe while preserving order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            out.append(u)
            seen.add(u)

    return out

In [6]:
urls_2025 = scrape_portal_player_links(driver, "https://247sports.com/season/2025-football/transferportaltop/")
len(urls_2025), urls_2025[2:7]
# add tqdm to this
# remove those first two cbs links 

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

(3010,
 ['https://247sports.com/player/nico-iamaleava-46101236/college-289779',
  'https://247sports.com/player/isaiah-world-46100635/college-325778',
  'https://247sports.com/player/damon-wilson-ii-46114588/college-291463',
  'https://247sports.com/player/carson-beck-46053141/college-245988',
  'https://247sports.com/player/eric-singleton-jr-46134398/college-297869'])

In [7]:
urls_2024 = scrape_portal_player_links(driver, "https://247sports.com/season/2024-football/transferportaltop/")
len(urls_2024), urls_2024[2:7]
# add tqdm to this
# remove those first two cbs links 

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

(2007,
 ['https://247sports.com/player/caleb-downs-46097147/college-290602',
  'https://247sports.com/player/kadyn-proctor-46099829/college-311211',
  'https://247sports.com/player/walter-nolen-46083769/college-281784',
  'https://247sports.com/player/isaiah-bond-46097889/college-281907',
  'https://247sports.com/player/evan-stewart-46096676/college-275133'])

In [12]:
all_urls = list(dict.fromkeys(urls_2025 + urls_2024))
print("total portal urls:", len(all_urls))

total portal urls: 5015


In [23]:
def get_text(driver, xpath, timeout=10):
    el = WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xpath)))
    return el.text.strip()

def body_text(driver):
    return driver.find_element(By.TAG_NAME, "body").text

def extract_player_id_247(url: str):
    m = re.search(r"/player/[^/]+-(\d+)", url)
    return int(m.group(1)) if m else None

def parse_int_from_text(s: str) -> Optional[int]:
    m = re.search(r"(\d{1,6})", s.replace(",", ""))
    return int(m.group(1)) if m else None

def parse_float_from_text(s: str) -> Optional[float]:
    m = re.search(r"(\d+\.\d+|\d+)", s.replace(",", ""))
    return float(m.group(1)) if m else None

def extract_transfer_block(txt: str, pos: str | None):
    if "247SPORTS TRANSFER RANKINGS" not in txt:
        return None, None, None, None
    m = re.search(r"247SPORTS TRANSFER RANKINGS\s+(\d+)\s+\((\d{4})\)", txt)
    rating = int(m.group(1)) if m else None
    year = int(m.group(2)) if m else None
    ovr = int(re.search(r"\bOVR\s+(\d+)\b", txt).group(1)) if re.search(r"\bOVR\s+(\d+)\b", txt) else None

    # pos rank: restrict search to after transfer header
    after = txt.split("247SPORTS TRANSFER RANKINGS", 1)[1]
    pos_rank = None
    if pos:
        m2 = re.search(rf"\b{re.escape(pos)}\s+(\d+)\b", after)
        pos_rank = int(m2.group(1)) if m2 else None
    return rating, year, ovr, pos_rank

DATE_LINE_RE = re.compile(r"[A-Za-z]{3}\s+\d{1,2},\s+20\d{2}:\s*\w+")

def _has_date_lines(driver) -> bool:
    try:
        txt = driver.find_element(By.TAG_NAME, "body").text
        return DATE_LINE_RE.search(txt) is not None
    except Exception:
        return False

def expand_timeline(driver, max_clicks=40):
    # Only click "Load more" buttons INSIDE the timeline area
    for _ in range(max_clicks):
        try:
            btn = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//*[@id='timeline']//button[contains(.,'Load more') or contains(.,'Load More')]")
                )
            )
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.15)
            driver.execute_script("arguments[0].click();", btn)  # JS click avoids interception
            time.sleep(0.5)
        except Exception:
            break


def open_timeline(driver, tries=3):
    # Goal: end with timeline date lines present in body text.
    for attempt in range(tries):
        # Ensure page is loaded
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//h1")))

        # If date lines already exist, we're done
        if _has_date_lines(driver):
            return True

        # 1) Jump to #timeline (hash link if available)
        try:
            a = driver.find_element(By.XPATH, "//a[@href='#timeline' and normalize-space()='Timeline']")
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", a)
            time.sleep(0.15)
            driver.execute_script("arguments[0].click();", a)
        except Exception:
            pass

        # 2) Scroll to the actual timeline container if present
        try:
            section = driver.find_element(By.CSS_SELECTOR, "#timeline")
            driver.execute_script("arguments[0].scrollIntoView({block:'start'});", section)
            time.sleep(0.25)
        except Exception:
            pass

        # 3) Expand entries
        expand_timeline(driver)

        # 4) Poll for date lines (don’t use a brittle WebDriverWait lambda here)
        t0 = time.time()
        while time.time() - t0 < 10:
            if _has_date_lines(driver):
                return True
            time.sleep(0.25)

        # Retry strategy: small scroll jiggle + (optional) soft refresh by reloading same URL
        driver.execute_script("window.scrollBy(0, 300);")
        time.sleep(0.25)
        driver.execute_script("window.scrollBy(0, -300);")
        time.sleep(0.25)

        if attempt < tries - 1:
            driver.get(driver.current_url)  # reload and try again

    return False


def parse_date_line(line: str):
    m = re.match(r"^([A-Za-z]{3}\s+\d{1,2},\s+\d{4}):\s*(.+)$", line.strip())
    if not m:
        return None, None
    dt = datetime.strptime(m.group(1), "%b %d, %Y")
    kind = m.group(2).strip()
    return dt, kind

def get_school_from_event_row(row_el):
    """
    Given a timeline event row element, return the best school name found.
    Prefer anchor text, fallback to last words after 'to'/'at'.
    """
    # Anchor text is usually full (not truncated like body.text)
    anchors = row_el.find_elements(By.XPATH, ".//a[normalize-space()]")
    # Often the school is the last anchor in the sentence
    if anchors:
        txts = [a.text.strip() for a in anchors if a.text and a.text.strip()]
        if txts:
            return txts[-1]

    # fallback: raw text
    t = row_el.text.strip()
    m = re.search(r"\b(commits to|enrolls at)\s+(.+)$", t, flags=re.IGNORECASE)
    return m.group(2).strip() if m else None

def get_timeline_rows(driver):
    """
    Returns list of candidate timeline event row elements.
    We locate by the presence of a date prefix like 'Dec 23, 2024:' somewhere in the row.
    """
    # This is intentionally broad and robust.
    return driver.find_elements(
        By.XPATH,
        "//*[contains(.,', 20') and contains(.,':')][.//*[contains(.,'Transfer') or contains(.,'Enrolled') or contains(.,'Commit') or contains(.,'Signed')]]"
    )

def extract_origin_destination_from_timeline(driver, window_start, window_end):
    """
    Returns: origin_school, destination_school, commit_dt
    """
    open_timeline(driver)
    rows = get_timeline_rows(driver)

    # Build structured rows with dt/kind + element
    parsed = []
    for r in rows:
        # find the first line inside the row that looks like a date line
        lines = [ln.strip() for ln in r.text.splitlines() if ln.strip()]
        dt, kind = None, None
        for ln in lines[:3]:
            dt, kind = parse_date_line(ln)
            if dt:
                break
        if dt and kind:
            parsed.append((dt, kind, r))

    # Sort newest first
    parsed.sort(key=lambda x: x[0], reverse=True)

    # Find destination commit/enroll event in window
    dest = None
    dest_dt = None
    dest_row = None
    for dt, kind, r in parsed:
        if not (window_start <= dt <= window_end):
            continue
        if kind.lower() == "transfer":
            # need commit/enroll row text under it, so just use row element
            row_text = r.text.lower()
            if "commits to" in row_text or "enrolls at" in row_text:
                dest = get_school_from_event_row(r)
                dest_dt = dt
                dest_row = r
                break

    if not dest_dt:
        return None, None, None

    # Find origin: nearest prior enrolled (preferred) before dest_dt
    origin = None
    for dt, kind, r in parsed:
        if dt >= dest_dt:
            continue
        if kind.lower() == "enrolled":
            origin = get_school_from_event_row(r)
            break

    # Fallback: prior commit/signed if no enrolled found
    if not origin:
        for dt, kind, r in parsed:
            if dt >= dest_dt:
                continue
            if kind.lower() in {"commit", "signed"}:
                origin = get_school_from_event_row(r)
                break

    return origin, dest, dest_dt


def count_stars(driver):
    selectors = [
        "[aria-label*='star']",                 # accessibility labels
        "svg[title*='star' i]",                 # svg title
        "svg[class*='star' i]",                 # svg class contains star
        "i[class*='star' i]",                   # font icons
        ".rankings-stars svg",                  # common wrapper
        ".stars svg",
    ]
    for sel in selectors:
        els = driver.find_elements(By.CSS_SELECTOR, sel)
        n = len(els)
        if 1 <= n <= 7:
            return n
    return None



from urllib.parse import urlparse

def get_ncaa_institutions(driver):
    """
    Returns canonical NCAA institution names from the institution dropdown.
    Works even when list is hidden by reading from hrefs.
    Example: ["Auburn", "Georgia Tech"]
    """
    # ensure dropdown is opened at least once
    try:
        btn = driver.find_element(By.CSS_SELECTOR, "button[data-js='institution-selector']")
        btn.click()
        time.sleep(0.2)
    except Exception:
        pass

    names = []

    # pull from hrefs (reliable) + fallback to text
    els = driver.find_elements(By.CSS_SELECTOR, "a.profile-card__institution-list-link")
    for a in els:
        href = (a.get_attribute("href") or "").strip()
        txt = (a.text or "").strip()

        # Prefer href pattern: .../college-123456  (NCAA entries)
        if "/college-" in href:
            # The anchor text may be empty if hidden; parse school from URL path segment after /player/.../
            # Example href: https://247sports.com/player/eric-singleton-jr-46134398/college-328493
            # We can't get school directly from URL, so use txt if present
            if txt:
                # "Auburn (NCAA)" -> "Auburn"
                if "(NCAA)" in txt:
                    names.append(txt.replace("(NCAA)", "").strip())
                else:
                    # sometimes just "Auburn"
                    names.append(txt.strip())
            else:
                # fallback: use the page itself to map college-IDs to school names
                # We'll collect the college IDs now; mapping happens below.
                pass

    # If text-based names worked, dedupe and return
    names = [n for n in names if n]
    if names:
        seen, out = set(), []
        for n in names:
            if n not in seen:
                out.append(n); seen.add(n)
        return out

    # HARD fallback (works even when hidden): use institution block HTML and parse the anchor text from outerHTML
    # outerHTML includes the text even if not visible to Selenium .text sometimes.
    names = []
    for a in els:
        html = a.get_attribute("outerHTML") or ""
        m = re.search(r">([^<]+)\(NCAA\)<", html)
        if m:
            names.append(m.group(1).strip())
        else:
            m2 = re.search(r">([^<]+)</a>", html)
            if m2 and "HS" not in m2.group(1):
                names.append(m2.group(1).replace("(NCAA)","").strip())

    names = [n for n in names if n]
    seen, out = set(), []
    for n in names:
        if n not in seen:
            out.append(n); seen.add(n)
    return out


def extract_school_fragment(event_text: str):
    """
    From a timeline description line, extract the trailing school fragment.
    Handles: "commits to X", "enrolls at X"
    """
    m = re.search(r"\b(commits to|enrolls at)\s+(.+)$", event_text, flags=re.IGNORECASE)
    if not m:
        return None
    frag = m.group(2).strip()
    return frag

def resolve_school_fragment(fragment: str, candidates: list[str]):
    """
    Resolve a possibly-truncated fragment ("Georgia Tech Yellow...") to a canonical school ("Georgia Tech").
    """
    if not fragment or not candidates:
        return None

    frag = fragment.replace("...", "").strip()

    # exact containment check (fast path)
    frag_low = frag.lower()
    for c in candidates:
        if c.lower() in frag_low or frag_low in c.lower():
            return c

    # fuzzy match
    best = process.extractOne(frag, candidates, scorer=fuzz.partial_ratio)
    if best and best[1] >= 80:
        return best[0]
    return None

def infer_origin_dest_from_timeline_events(events, candidates, window_start, window_end):
    """
    events: list[TL] newest-first (your parse yields newest-first)
    candidates: canonical NCAA schools for the player (from institution list)
    returns origin, destination, commit_dt
    """
    # destination = newest Transfer commit/enroll within window
    dest_frag = None
    dest_dt = None

    for e in events:
        if not (window_start <= e.date <= window_end):
            continue
        if e.kind.lower() == "transfer":
            frag = extract_school_fragment(e.text)
            if frag:
                dest_frag = frag
                dest_dt = e.date
                break

    if not dest_dt:
        return None, None, None

    dest = resolve_school_fragment(dest_frag, candidates)

    # origin = nearest prior Enrolled (preferred) before dest_dt
    origin_frag = None
    for e in events:
        if e.date >= dest_dt:
            continue
        if e.kind.lower() == "enrolled":
            frag = extract_school_fragment(e.text)
            if frag:
                origin_frag = frag
                break

    # fallback origin = prior Commit/Signed before dest_dt
    if not origin_frag:
        for e in events:
            if e.date >= dest_dt:
                continue
            if e.kind.lower() in {"commit", "signed"}:
                frag = extract_school_fragment(e.text)
                if frag:
                    origin_frag = frag
                    break

    origin = resolve_school_fragment(origin_frag, candidates)

    return origin, dest, dest_dt

def timeline_lines(txt: str):
    lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    # keep only timeline-ish region
    u = [ln.upper() for ln in lines]
    if "TIMELINE" in u:
        start = u.index("TIMELINE")
        lines = lines[start:]
    # cut off when you hit unrelated sections
    for stop_token in ["IN PICTURES", "ARTICLES", "CBS SPORTS DIGITAL"]:
        if stop_token in [x.upper() for x in lines]:
            stop = [x.upper() for x in lines].index(stop_token)
            lines = lines[:stop]
            break
    return lines

@dataclass
class TL:
    date: datetime
    kind: str
    text: str

DATE_RE = re.compile(r"^([A-Za-z]{3}\s+\d{1,2},\s+\d{4}):\s*(.+)$")

def parse_timeline_events(lines):
    events = []
    i = 0
    while i < len(lines):
        m = DATE_RE.match(lines[i])
        if m:
            dt = datetime.strptime(m.group(1), "%b %d, %Y")
            kind = m.group(2).strip()  # Transfer / Enrolled / Commit / Signed
            text = lines[i+1].strip() if i+1 < len(lines) else ""
            events.append(TL(date=dt, kind=kind, text=text))
            i += 2
        else:
            i += 1
    return events

In [24]:
def scrape_hs_recruiting_page(driver, hs_url: str):
    driver.get(hs_url)
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//h1")))
    txt = body_text(driver)

    # ---- Normalize into clean lines ----
    raw_lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    # Remove css/garbage lines like ".st0{fill-rule:...}"
    lines = []
    for ln in raw_lines:
        if ln.startswith(".") and "{" in ln and "}" in ln:
            continue
        if "fill-rule" in ln or "clip-rule" in ln or "st0{" in ln:
            continue
        lines.append(ln)

    # Helper: find first index of a token (case-insensitive)
    def idx_of(token: str):
        token_u = token.upper()
        for i, ln in enumerate(lines):
            if ln.upper() == token_u:
                return i
        return None

    # ---- CLASS ----
    hs_class = None
    i = idx_of("CLASS")
    if i is not None and i + 1 < len(lines) and lines[i+1].isdigit():
        hs_class = int(lines[i+1])

    # ---- 247SPORTS block (non-composite) ----
    hs_rating_247 = None
    hs_natl_rank = None
    hs_pos = None
    hs_pos_rank = None

    i247 = idx_of("247SPORTS")
    if i247 is not None:
        # Next numeric line after 247SPORTS is rating
        for j in range(i247 + 1, min(i247 + 10, len(lines))):
            if re.fullmatch(r"\d{1,3}", lines[j]):
                hs_rating_247 = int(lines[j])
                break

    inatl = idx_of("NATL.")
    if inatl is not None:
        # Next numeric line after NATL. is natl rank
        for j in range(inatl + 1, min(inatl + 6, len(lines))):
            if re.fullmatch(r"\d{1,6}", lines[j]):
                hs_natl_rank = int(lines[j])
                # Next token+number pair after natl is position + pos rank (skip state line later)
                # Find first [A-Z]{1,4} then number
                for k in range(j + 1, min(j + 10, len(lines))):
                    if re.fullmatch(r"[A-Z]{1,4}", lines[k]):
                        # avoid "CA", "TX" etc being misread as position only if it's two letters AND appears after QB line
                        # but for HS pages this first one after natl is usually position (QB/WR/RB/ATH/CB/S/LB/EDGE/OT/IOL/DL)
                        if k + 1 < len(lines) and re.fullmatch(r"\d{1,6}", lines[k+1]):
                            hs_pos = lines[k].upper()
                            hs_pos_rank = int(lines[k+1])
                            break
                break

    # ---- 247SPORTS COMPOSITE® block ----
    composite_rating = None
    composite_natl = None
    composite_pos = None
    composite_pos_rank = None

    # Find "247SPORTS COMPOSITE®" or "247SPORTS COMPOSITE"
    icomp = None
    for token in ["247SPORTS COMPOSITE®", "247SPORTS COMPOSITE"]:
        icomp = idx_of(token)
        if icomp is not None:
            break

    if icomp is not None:
        # Next float after comp token is rating (0.9981)
        for j in range(icomp + 1, min(icomp + 12, len(lines))):
            if re.fullmatch(r"\d\.\d+", lines[j]):
                composite_rating = float(lines[j])
                break
        # Find NATL. after composite header (search forward)
        for j in range(icomp, min(icomp + 30, len(lines))):
            if lines[j].upper() == "NATL.":
                # next numeric is composite natl rank
                for k in range(j + 1, min(j + 6, len(lines))):
                    if re.fullmatch(r"\d{1,6}", lines[k]):
                        composite_natl = int(lines[k])
                        # next token+number is composite pos + rank
                        for t in range(k + 1, min(k + 12, len(lines))):
                            if re.fullmatch(r"[A-Z]{1,4}", lines[t]) and t + 1 < len(lines) and re.fullmatch(r"\d{1,6}", lines[t+1]):
                                composite_pos = lines[t].upper()
                                composite_pos_rank = int(lines[t+1])
                                break
                        break
                break

    # ---- Stars (we'll set later once we pick a reliable selector) ----
    hs_stars = count_stars(driver)

    return {
        "hs_class": hs_class,
        "hs_rating_247": hs_rating_247,
        # "hs_natl_rank": hs_natl_rank,
        "hs_pos": hs_pos,
        # "hs_pos_rank": hs_pos_rank,
        "composite_rating": composite_rating,
        "composite_natl_rank": composite_natl,
        # "composite_pos": composite_pos,
        "composite_pos_rank": composite_pos_rank,
        "hs_stars": hs_stars,
        "source_hs_url": hs_url
    }


In [25]:
def scrape_player(driver, player_url: str):
    driver.get(player_url)
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//h1")))
    txt = body_text(driver)

    name = get_text(driver, "//h1", timeout=10)
    pid = extract_player_id_247(player_url)

    # Position from "POS QB"
    pos = None
    m_pos = re.search(r"\bPOS\s+([A-Z]{1,4})\b", txt)
    if m_pos:
        pos = m_pos.group(1)

    # Prospect info (HS, City)
    hs_name = re.search(r"HIGH SCHOOL\s+([^\n]+)", txt).group(1).strip() if "HIGH SCHOOL" in txt else None
    city = re.search(r"CITY\s+([^\n]+)", txt).group(1).strip() if "CITY" in txt else None
    hs_city, hs_state = None, None
    if city and "," in city:
        hs_city = city.split(",")[0].strip()
        hs_state = city.split(",")[1].strip()

    # Transfer rankings block (your existing parser)
    t_rating, t_year, t_ovr, t_posrank = extract_transfer_block(txt, pos)

    # HS recruiting URL
    hs_url = None
    try:
        hs_url = driver.find_element(By.XPATH, "//a[contains(., 'View recruiting profile')]").get_attribute("href")
    except Exception:
        anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/high-school-')]")
        if anchors:
            hs_url = anchors[0].get_attribute("href")

    hs_data = {}
    if hs_url:
        hs_url = hs_url.split("?")[0].rstrip("/")
        hs_data = scrape_hs_recruiting_page(driver, hs_url)

    # ---- Timeline origin/destination ----
    # --- TIMELINE DEBUG CHECK (temporary, remove later) ---
    open_timeline(driver)
    txt2 = body_text(driver)

    lines = timeline_lines(txt2)
    events = parse_timeline_events(lines)

    # HARD ASSERT: we must see Transfer + Enrolled
    if not any(e.kind == "Transfer" for e in events):
        raise RuntimeError("Timeline parsed but no Transfer events found")

    if not any(e.kind == "Enrolled" for e in events):
        raise RuntimeError("Timeline parsed but no Enrolled events found")

    # Optional debug print
    if DEBUG:
        print(f"\nTIMELINE CHECK for {name}")
        for e in events[:8]:
            print(e.date.strftime("%Y-%m-%d"), e.kind, "|", e.text)

    # canonical NCAA schools from institution list (may be hidden; click dropdown if empty)
    cands = get_ncaa_institutions(driver)
    if DEBUG:
        print("INSTITUTION CANDIDATES:", cands)
    if not cands:
        try:
            inst_btn = driver.find_element(By.CSS_SELECTOR, "button[data-js='institution-selector']")
            inst_btn.click()
            time.sleep(0.3)
        except Exception:
            pass
        cands = get_ncaa_institutions(driver)

    origin, dest, commit_dt = infer_origin_dest_from_timeline_events(
        events,
        candidates=cands,
        window_start=datetime(2024, 11, 15),
        window_end=datetime(2025, 8, 1),
    )

    portal_season_year = None
    if commit_dt:
        portal_season_year = commit_dt.year + 1 if commit_dt.month == 12 else commit_dt.year


    # Stars derived from rating (fast + consistent)
    def stars_from_rating(r):
        if r is None: return None
        if r >= 98: return 5
        if r >= 90: return 4
        if r >= 80: return 3
        if r <= 80: return 2
        return 0

    transfer_stars = stars_from_rating(t_rating)
    hs_stars = stars_from_rating(hs_data.get("hs_rating_247")) if hs_data else None

    return {
        "id_247": pid,
        "name": name,
        "pos_247": pos,
        "hs_name": hs_name,
        "hs_city": hs_city,
        "hs_state": hs_state,

        "transfer_rating": t_rating,
        "transfer_year": portal_season_year or t_year,   # prefer timeline-derived portal season year
        "transfer_ovr_rank": t_ovr,
        "transfer_pos_rank": t_posrank,
        "transfer_stars": transfer_stars,
        "transfer_origin": origin,
        "transfer_destination": dest,

        **hs_data,
        "hs_stars": hs_stars,
        "source_hs_url": hs_url,
        "source_player_url": player_url
    }


In [11]:
# DEBUG = False

# p = scrape_player(driver, 'https://247sports.com/player/eric-singleton-jr-46134398/college-297869/')
# p

# # double check stars
# # stars
# # timeline events/start/end
# # hs_pos_rank & hs_natl_rank are 247 Composite

In [ ]:
for u in tqdm(url):
    try:
        d = scrape_player(driver, u)
        rows.append(d)
        append_cache(d)
    except Exception as e:
        print("FAIL", u, e)

portal_df = pd.DataFrame(rows)

In [26]:
import os, json, pandas as pd
from tqdm import tqdm

CACHE_PATH = "data/portal_cache.jsonl"   # one JSON per line

def load_cache(path=CACHE_PATH):
    if not os.path.exists(path):
        return {}
    out = {}
    with open(path, "r") as f:
        for line in f:
            try:
                d = json.loads(line)
                out[d["source_player_url"]] = d
            except:
                pass
    return out

def append_cache(d, path=CACHE_PATH):
    with open(path, "a") as f:
        f.write(json.dumps(d) + "\n")

cache = load_cache()
# todo = [u for u in all_urls if u.rstrip("/") not in cache]
todo = [u for u in urls_2025[2:] if u.rstrip("/") not in cache]
print("cached:", len(cache), "todo:", len(todo))

rows = list(cache.values())
for u in tqdm(todo):
    try:
        d = scrape_player(driver, u)
        rows.append(d)
        append_cache(d)
    except Exception as e:
        print("FAIL", u, e)

portal_df = pd.DataFrame(rows)


cached: 0 todo: 3008


  3%|▎         | 81/3008 [00:00<00:03, 805.01it/s]

FAIL https://247sports.com/player/nico-iamaleava-46101236/college-289779 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/isaiah-world-46100635/college-325778 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/damon-wilson-ii-46114588/college-291463 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new co

  6%|▌         | 184/3008 [00:00<00:03, 933.49it/s]

FAIL https://247sports.com/player/jayviar-suggs-46149118/college-315452 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/tyler-onyedim-46085500/college-267576 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))


...

FAIL https://247sports.com/player/jake-woods-46046607/college-337101 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/alex-branch-46085692/college-271528 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/rickey-hyatt-jr-46080419/college-287415 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connect

 95%|█████████▌| 2869/3008 [00:02<00:00, 976.15it/s]

FAIL https://247sports.com/player/javen-nicholas-46158171/college-337041 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/brandon-best-46115416/college-284071 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/ken-willis-46112008/college-286045 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connect

100%|██████████| 3008/3008 [00:03<00:00, 961.47it/s]

FAIL https://247sports.com/player/micah-hudson-46114316/college-311143 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/julian-neal-46100114/college-272102 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connection: [Errno 61] Connection refused"))
FAIL https://247sports.com/player/tanner-koziol-46097332/college-305279 HTTPConnectionPool(host='localhost', port=63005): Max retries exceeded with url: /session/8f00cccee383e17bfa8cb9cb2308c426/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63005): Failed to establish a new connect

In [6]:
def test_timeline(driver, player_url):
    driver.get(player_url)

    ok = open_timeline(driver, tries=3)
    if not ok:
        raise RuntimeError("Timeline did not load date lines after retries")

    events = parse_timeline_events(
        timeline_lines(body_text(driver))
    )

    for e in events[:10]:
        print(e.date.strftime("%Y-%m-%d"), e.kind, "|", e.text)

    return events


events = test_timeline(driver, "https://247sports.com/player/eric-singleton-jr-46134398/college-297869/")
len(events)

2024-12-23 Transfer | Eric Singleton Jr. commits to Auburn Tigers
2024-12-09 Transfer | Eric Singleton Jr. entered the transfer portal
2023-06-07 Enrolled | Eric Singleton Jr. enrolls at Georgia Tech Yellow...
2022-12-21 Commit | Eric Singleton Jr. commits to Georgia Tech Yellow...
2022-12-21 Signed | Eric Singleton Jr. signs letter of intent to...


5

In [ ]:
driver.quit()

In [ ]:
# -----------------------------
# Excel fill logic
# -----------------------------
def build_lookup(players):
    by_id = {p.id_247: p for p in players if p.id_247}
    by_namepos = {(norm_name(p.name), (p.position or "").upper()): p for p in players}
    return by_id, by_namepos

def match_player(row_player_id, row_name, row_pos, by_id, by_namepos):
    if pd.notna(row_player_id):
        pid = int(row_player_id)
        if pid in by_id:
            return by_id[pid]
    key = (norm_name(row_name), (row_pos or "").upper())
    if key in by_namepos:
        return by_namepos[key]
    # optional fuzzy fallback here...
    return None

    # fuzzy within same position
    candidates = [(k, v) for k, v in lookup.items() if k[1] == (row_pos or "").upper()]
    if not candidates:
        return None
    names = [k[0] for k, _ in candidates]
    best = process.extractOne(norm_name(row_name), names, scorer=fuzz.WRatio)
    if not best or best[1] < 92:
        return None
    matched_norm = best[0]
    for (k_norm, k_pos), v in lookup.items():
        if k_norm == matched_norm and k_pos == (row_pos or "").upper():
            return v
    return None

def fill_excel(in_path: str, out_path: str, scraped: List[Player247]) -> None:
    df = pd.read_excel(in_path)

    # Only fill for 100+ snaps
    target = df["pff_snaps"].fillna(0) >= 100
    by_id, by_namepos = build_lookup(scraped)
    row_pid = row.get("player_id")
    p = match_player(row_pid, row_name, row_pos, by_id, by_namepos)

    # columns
    F = "247/On3 Position"
    G = "High School, City, State"
    H = "247 'Exp' (aka High School Class)"
    I = "H/S Stars"
    J = "H/S Rating"
    K = "H/S National Rank"
    L = "H/S Position Rank"

    M = "Transfer Year"
    N = "Transfer Origin"
    O = "Origin P4 / G5 / Non-FBS"
    P = "Transfer Destination"
    Q = "Destination P4 / G5 / Non-FBS"
    R = "Transfer Stars"
    S = "Transfer Rating"
    T = "Transfer Overall Rank"
    U = "Transfer Position Rank"

    for idx in tqdm(df.index[target], desc="Filling rows"):
        row = df.loc[idx]
        row_name = row.get("full_name", "")
        row_pos = str(row.get("position", "")).upper()  # your sheet has 'position'
        p = match_player(row_name, row_pos, lookup)
        if not p:
            continue

        df.at[idx, F] = p.position
        if p.hs_name and p.hs_city and p.hs_state:
            df.at[idx, G] = f"{p.hs_name}, {p.hs_city}, {p.hs_state}"
        df.at[idx, H] = p.hs_exp or p.hs_class
        df.at[idx, I] = p.hs_stars
        df.at[idx, J] = p.hs_rating_247
        df.at[idx, K] = p.hs_natl_rank
        df.at[idx, L] = p.hs_pos_rank

        # Transfer: only fill if you have transfer info (else leave blank M:U)
        if p.transfer_rating or p.transfer_ovr_rank or p.transfer_pos_rank:
            df.at[idx, M] = p.transfer_year
            df.at[idx, N] = p.transfer_origin
            df.at[idx, O] = classify_fbs(p.transfer_origin)
            df.at[idx, P] = p.transfer_destination
            df.at[idx, Q] = classify_fbs(p.transfer_destination)
            df.at[idx, R] = p.transfer_stars
            df.at[idx, S] = p.transfer_rating
            df.at[idx, T] = p.transfer_ovr_rank
            df.at[idx, U] = p.transfer_pos_rank

    df.to_excel(out_path, index=False)


def main():
    portal_urls = [
        "https://247sports.com/season/2024-football/transferportaltop/",
        "https://247sports.com/season/2025-football/transferportaltop/",
    ]

    driver = make_driver(headless=True)
    try:
        all_player_urls = []
        for url in portal_urls:
            links = scrape_portal_player_links(driver, url)
            all_player_urls.extend(links)

        # Deduplicate
        all_player_urls = list(dict.fromkeys(all_player_urls))

        scraped: List[Player247] = []
        for u in tqdm(all_player_urls, desc="Scraping player pages"):
            try:
                scraped.append(scrape_player_pages(driver, u))
            except Exception:
                continue

        fill_excel(
            in_path="2024-2025 Player Database v2.xlsx",
            out_path="2024-2025 Player Database v2_FILLED.xlsx",
            scraped=scraped,
        )
    finally:
        driver.quit()

if __name__ == "__main__":
    main()